# Deep Learning Example
Using Student_performance_data.csv from Kaggle: https://www.kaggle.com/datasets/rabieelkharoua/students-performance-dataset

In [1]:
import torch 
print(torch.__version__)

2.11.0+cu130


## Data Pre-processing
Why pre-process data? Handle missing values to ensure the dataset is uniform (consistency), select features most relevant to the problem (relevance), and normalize the data scale (efficiency).
### Missing Data
Common strategies include removing rows/cols with missing data (removal); replacing missing data with mean, median, or mode (imputation); or estimating missing data based on other data (interpolation).
### Scaling and Normalization
Common strategies include scaling so all values fit within a range, typically [0, 1] (min-max scaling) and centering data around a mean with a unit of standard deviation (standardization).
### Splitting Data
Split the dataset to training and testing sets. Training sets train a model while testing sets evaluate performance. If a model trains on all the data, the model will be overfitted to the data, memorizing answers. Use a split of 80-20 for training-to-testing.

In [ ]:
import pandas as pd

data = pd.read_csv('./data/Student_performance_data.csv')

The data is loaded like a relational data table. We can use header names to pick columns and row numbers to specify subsets in the data.

In [6]:
# grab columns with indexing
#print(data['StudyTimeWeekly']) 
# grab rows with iloc method
#print(data.iloc[5]) 

First we can check if the data is null. This dataset should be pretty full, but if there's missing data, we can use imputation.

In [ ]:
print(data.isnull().sum())
# Impute missing values with the mean of each column
#data.fillna(data.mean(), inplace=True)

StudentID            0
Age                  0
Gender               0
Ethnicity            0
ParentalEducation    0
StudyTimeWeekly      0
Absences             0
Tutoring             0
ParentalSupport      0
Extracurricular      0
Sports               0
Music                0
Volunteering         0
GPA                  0
GradeClass           0
dtype: int64


Pandas doesn't have normalization built-in, but scikit-learn does.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Select features and target
features = data.drop('StudentID', axis=1)
features = features.drop("GradeClass", axis=1)
labels = data["GradeClass"].astype(int)

# Standardize the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

Now the data can be split into training and testing sets. The scaled_features is the 2D array or DataFrame containing the standardized features used for training the model. The test_size specifies how much of the data set should be used for testing (0.2 = 20%). The random_state is the seed for the random number generator. Specifying the seed ensures the split is reproducible. 

In [9]:
from sklearn.model_selection import train_test_split

# Split the data
training_f, testing_f, training_l, testing_l = train_test_split(scaled_features, labels, test_size=0.2, random_state=42)

Next step is to convert the sets to PyTorch tensors.

In [13]:
# Convert to PyTorch tensors
training_f = torch.tensor(training_f, dtype=torch.float32)
testing_f = torch.tensor(testing_f, dtype=torch.float32)
training_l = torch.tensor(training_l.to_numpy(), dtype=torch.long) # long = whole numbers
testing_l = torch.tensor(testing_l.to_numpy(), dtype=torch.long)

Next step is to structure the data for efficient loading and batching. PyTorch has `Dataset` and `Dataloader` classes for this. TensorDataset creates a tensor of 2 indexed tensors. Index 0 holds features and index 1 holds the label (`python [ [[features], [labels]], [[features], [labels]], [[features], [labels]], ]`). DataLoader creates an iterable object that takes the original TensorDataset and shuffles the nested samples whenever it's called.

In [14]:
from torch.utils.data import DataLoader, TensorDataset

# Create TensorDatasets
training_dataset = TensorDataset(training_f, training_l) # aligns features and labels
testing_dataset = TensorDataset(testing_f, testing_l)

# Create DataLoaders
training_loader = DataLoader(training_dataset, batch_size=32, shuffle=True)
testing_loader = DataLoader(testing_dataset, batch_size=32, shuffle=True)

## Softmax
Softmax is an activation function used in the output layer of a neural network for multi-class classification. Softmax converts the raw output scores or logits into probabilities, the sum of them add up to 1.

## Building a Multiclassification Model using PyTorch
PyTorch uses `torch.nn` to build neural networks. It provides layers, loss functions, and other tools needed. `nn.Linear` applies linear transformation to the input data. It has parameters called weights and biases used during training. Capabilities are linear transformations of the input data. Limitations are for non-linear relationships in the data. Weaknesses model complex patterns without activation functions. Type is a hidden layer placed between input and output layers. Data Transfer transforms data through the weighted sum and adds biases.

In [41]:
import torch.nn as n

class StudentPerformanceModel(n.Module): # Module there is a requirement for the existence of a method named forward

  def __init__(self):
    super(StudentPerformanceModel, self).__init__()
    # ly = layer
    self.ly1 = n.Linear(13,30) #=> 30
    self.ly2 = n.Linear(30,25) #=> 25
    self.ly3 = n.Linear(25,20)
    self.ly4 = n.Linear(20,15)
    self.ly5 = n.Linear(15,10)
    self.ly6 = n.Linear(10,5)
    self.act = n.Softmax(dim=1) #tgt is an int

  def forward(self, data):
    pass1 = self.ly1(data)
    pass2 = self.ly2(pass1)
    pass3 = self.ly3(pass2)
    pass4 = self.ly4(pass3)
    pass5 = self.ly5(pass4)
    pass6 = self.ly6(pass5)
    return self.act(pass6)

model = StudentPerformanceModel()

During training, weights and biases are adjusted to minimize loss functions. Here's how to see the weights and biases:

In [35]:
# View weights and biases
# for name, param in model.named_parameters():
#     if param.requires_grad:
#         print(name, param.data)

Model predictions are interpreted by finding a class with the highest probability. 

In [ ]:
# model.train()
# for epoch in range(10):
#     for features, labels in training_loader:
#         output = model(features)
#         _, predicted = torch.max(output.data, 1)

tensor([0.2242, 0.2235, 0.2293, 0.2217, 0.2194, 0.2165, 0.2207, 0.2165, 0.2226,
        0.2183, 0.2201, 0.2201, 0.2250, 0.2253, 0.2222, 0.2210, 0.2191, 0.2211,
        0.2233, 0.2275, 0.2244, 0.2188, 0.2284, 0.2219, 0.2201]) tensor([4, 4, 4, 4, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 4, 4, 4, 4, 4, 4,
        4])


Cross Entropy Loss is a loss function for multi-class classification. It measures the performance of a classification model where output is either 0 or 1, yes or no. 

Optimizers update the model parameters to minimize the loss function. The optimal minimum is the point when loss is smallest. 

Learning rate and momentum are parameters that affect the process. The learning rate is the step size of the parameter updates. A high learning rate can overshoot the minimum while a low learning rate takes too long. 

Momentum helps accelerate gradients to the right direction, speeding up converence.

In [42]:
from torch.optim import Adam
optimizer = Adam(model.parameters(), lr=0.01)

The back propagation or backward step is the process where the model adjusts its weights based on the error from the forward pass. 

Forward step passes input data through the model to get the output.

Backwards step computes the gradient of the loss with respect to each parameter, allowing optimizers to update model's weights.

In [43]:
from torch.optim import Adam

# Define loss function and optimizer
criterion = n.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001)

# Training loop
model.train()
for epoch in range(50):
  running_loss = 0
  for features, labels in training_loader:
    optimizer.zero_grad()
    output = model(features)
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()
  average_los = running_loss / len(training_loader)
  if not (epoch+1) % 10:
    print(f"Average Loss {average_los} @ epoc {epoch+1}")

Average Loss 1.0415410290161768 @ epoc 10
Average Loss 1.0394727557897567 @ epoc 20
Average Loss 1.0256131728490194 @ epoc 30
Average Loss 1.0249204903841018 @ epoc 40
Average Loss 1.0208099444707235 @ epoc 50


The evaluation loop uses `Accuracy` and `torchmetrics`:

In [44]:
from torchmetrics import Accuracy

# Initialize accuracy metric
accuracy = Accuracy(task='multiclass', num_classes=5)

# Evaluation loop
model.eval()
with torch.no_grad():
  for features, labels in testing_loader:
    output = model(features)
    _, predicted = torch.max(output.data, 1)
    accuracy.update(predicted, labels)

print(f"Accuracy: {accuracy.compute().item()}")

Accuracy: 0.8538622260093689
